<a href="https://colab.research.google.com/github/Yash1014-code/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yash1014-code/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1: The Random Forest showed measurable ranking signal for future CTR

The Week-5 Random Forest model used GSC and GA4 features available before the prediction period and was evaluated on the following month's CTR. The model achieved a Spearman rank correlation of approximately **0.174** on the March→April evaluation population, indicating a measurable but modest relationship between predicted and observed future CTR.

**Methodology question — where does the label come from?**
The target label is the future monthly CTR, calculated as April clicks divided by April impressions for the same client-content unit. The model features are constructed from March data, so the target occurs after the feature window. This separation reduces the risk of directly using future outcome information as an input.

**Methodology question — does the validation design carry the claim?**
The Week-5 evaluation uses a time-aware design: February features are used to predict March CTR for training, while March features are used to predict April CTR for evaluation. This supports a directional, future-oriented evaluation because the model is tested on a later time period rather than randomly mixing observations from the same period. However, the measured ranking signal is modest and should be interpreted as decision-support evidence rather than proof that the model will reliably predict CTR for every future page or client.

### Finding 2: The Random Forest ranked higher future-CTR pages than the Week-4 baseline at Top-K

On the March→April test population, the Random Forest produced higher mean future CTR among its highest-ranked pages than the Week-4 rule-based baseline. For example, at **Top-20**, the Random Forest achieved a mean future CTR of approximately **0.065**, compared with approximately **0.001** for the baseline.

**Methodology question — where does the label come from?**
The Top-K comparison uses the observed April CTR as the outcome. The ranking itself is generated from March-era features, while April CTR is used only after ranking to measure the result. Therefore, the future CTR is treated as the evaluation outcome rather than a model input.

**Methodology question — does the validation design carry the claim?**
The model and baseline are compared on the same March→April evaluation population, which makes the Top-K comparison useful as an observed benchmark. However, the result does not establish that using the Random Forest will cause CTR to increase. It only shows that, on this evaluated population, pages selected by the model had higher observed future CTR than pages selected by the baseline. A stricter grouped-client audit in Week 6 is therefore useful to test whether the observed signal remains when evaluating clients that were not seen during training.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week 5 already used a time-aware split: the model was trained on February features to predict March CTR, then evaluated on March features to predict April CTR. Therefore, the Week-6 audit does not replace a random split with a time split.

Instead, I reproduce the Week-5 time-aware evaluation as the before result and add a stricter grouped + time-aware evaluation as the after result. In the grouped audit, clients used for evaluation are completely held out from model training.

This tests whether the measured ranking signal transfers to unseen clients rather than only measuring performance on clients that were also present during training.

The grouped evaluation is intentionally stricter, so a lower score is not treated as a failure. It is evidence about how well the observed signal generalizes to unseen clients.

In [2]:
# Week 6 setup: imports and Hugging Face connection

from google.colab import userdata
from huggingface_hub import login
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Add your Hugging Face token "
        "to Colab Secrets with the name HF_TOKEN."
    )

login(token=HF_TOKEN, add_to_git_credential=False)

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute("""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face connection ready!")

Hugging Face connection ready!


In [3]:
# Load the same three months used in Week 5

def load_month(month):
    path = (
        f"hf://datasets/FlyRank/internship-warehouse/"
        f"fact_content_daily_performance/"
        f"month=2026-{month}/*.parquet"
    )

    query = f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        ga4_sessions,
        scroll_events
    FROM read_parquet('{path}')
    WHERE gsc_data_available IS TRUE
    """

    return con.execute(query).df()


feb = load_month("02")
mar = load_month("03")
apr = load_month("04")

print("February:", feb.shape)
print("March:", mar.shape)
print("April:", apr.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February: (2621783, 9)
March: (3611061, 9)
April: (3901060, 9)


In [4]:
# Convert daily data into client-content monthly data
# using the same aggregation logic as Week 5.

def aggregate_monthly(df):
    grouped = (
        df.groupby(
            ["client_hash_id", "content_hash_id"],
            as_index=False
        )
        .agg(
            gsc_impressions=("gsc_impressions", "sum"),
            gsc_clicks=("gsc_clicks", "sum"),
            gsc_sum_position=("gsc_sum_position", "sum"),
            ga4_sessions=("ga4_sessions", "sum"),
            scroll_events=("scroll_events", "sum")
        )
    )

    grouped["gsc_avg_position"] = (
        grouped["gsc_sum_position"] /
        grouped["gsc_impressions"].replace(0, pd.NA)
    )

    grouped["ctr"] = (
        grouped["gsc_clicks"] /
        grouped["gsc_impressions"].replace(0, pd.NA)
    )

    return grouped


feb_monthly = aggregate_monthly(feb)
mar_monthly = aggregate_monthly(mar)
apr_monthly = aggregate_monthly(apr)

print("February monthly rows:", len(feb_monthly))
print("March monthly rows:", len(mar_monthly))
print("April monthly rows:", len(apr_monthly))

February monthly rows: 153559
March monthly rows: 176738
April monthly rows: 194760


In [5]:
# Recreate the exact Week-5 time-aware setup:
#
# Training:
# February features -> March future CTR
#
# Testing:
# March features -> April future CTR

train_data = feb_monthly.merge(
    mar_monthly[
        ["client_hash_id", "content_hash_id", "ctr"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    suffixes=("_feb", "_mar")
)

test_data = mar_monthly.merge(
    apr_monthly[
        ["client_hash_id", "content_hash_id", "ctr"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    suffixes=("_mar", "_apr")
)

train_data = train_data.rename(
    columns={"ctr_mar": "future_ctr"}
)

test_data = test_data.rename(
    columns={"ctr_apr": "future_ctr"}
)

train_data = train_data.dropna(
    subset=["future_ctr"]
)

test_data = test_data.dropna(
    subset=["future_ctr"]
)

print("Training rows:", len(train_data))
print("Test rows:", len(test_data))
print("Training clients:", train_data["client_hash_id"].nunique())
print("Test clients:", test_data["client_hash_id"].nunique())

Training rows: 134238
Test rows: 158549
Training clients: 42
Test clients: 46


In [6]:
# Same features and target used in Week 5

features = [
    "gsc_avg_position",
    "gsc_impressions",
    "gsc_clicks",
    "ga4_sessions",
    "scroll_events"
]

target = "future_ctr"

print("Features:")
print(features)

print("\nTarget:")
print(target)

print("\nMissing values in training features:")
print(train_data[features].isna().sum())

print("\nMissing values in test features:")
print(test_data[features].isna().sum())

Features:
['gsc_avg_position', 'gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'scroll_events']

Target:
future_ctr

Missing values in training features:
gsc_avg_position    0
gsc_impressions     0
gsc_clicks          0
ga4_sessions        0
scroll_events       0
dtype: int64

Missing values in test features:
gsc_avg_position    0
gsc_impressions     0
gsc_clicks          0
ga4_sessions        0
scroll_events       0
dtype: int64


In [7]:
# Section 2: My model under an honest split (before/after)

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr
import numpy as np
import pandas as pd


# =========================================================
# 1. BEFORE: Reproduce the Week-5 time-aware evaluation
# =========================================================

reference_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

reference_model.fit(
    train_data[features],
    train_data[target]
)

reference_pred = reference_model.predict(
    test_data[features]
)

reference_metrics = {
    "Split": "Week-5 time-aware",
    "Test rows": len(test_data),
    "Clients": test_data["client_hash_id"].nunique(),
    "Base rate": test_data[target].mean(),
    "MAE": mean_absolute_error(
        test_data[target],
        reference_pred
    ),
    "RMSE": np.sqrt(
        mean_squared_error(
            test_data[target],
            reference_pred
        )
    ),
    "Spearman": spearmanr(
        test_data[target],
        reference_pred
    ).statistic
}


# =========================================================
# 2. AFTER: Grouped + time-aware evaluation
#    Hold out entire clients from model training
# =========================================================

# Use only clients that exist in both training
# and testing periods.

common_clients = sorted(
    set(train_data["client_hash_id"])
    .intersection(
        set(test_data["client_hash_id"])
    )
)

client_frame = pd.DataFrame({
    "client_hash_id": common_clients
})

print("Total common clients:", len(common_clients))


# Split clients instead of individual rows
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_client_idx, holdout_client_idx = next(
    gss.split(
        client_frame,
        groups=client_frame["client_hash_id"]
    )
)

train_clients = set(
    client_frame.iloc[train_client_idx]["client_hash_id"]
)

holdout_clients = set(
    client_frame.iloc[holdout_client_idx]["client_hash_id"]
)


# February -> March data for training
audit_train = train_data[
    train_data["client_hash_id"].isin(train_clients)
].copy()


# March -> April data for evaluation
audit_test = test_data[
    test_data["client_hash_id"].isin(holdout_clients)
].copy()


# =========================================================
# 3. Verify that clients do not overlap
# =========================================================

overlap = (
    set(audit_train["client_hash_id"])
    .intersection(
        set(audit_test["client_hash_id"])
    )
)

assert len(overlap) == 0

print("\nGrouped + time-aware split check")
print("----------------------------------")
print(
    "Training clients:",
    audit_train["client_hash_id"].nunique()
)
print(
    "Held-out clients:",
    audit_test["client_hash_id"].nunique()
)
print("Client overlap:", len(overlap))
print("Training rows:", len(audit_train))
print("Evaluation rows:", len(audit_test))


# =========================================================
# 4. Train the same Random Forest on the honest split
# =========================================================

audit_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

audit_model.fit(
    audit_train[features],
    audit_train[target]
)

audit_pred = audit_model.predict(
    audit_test[features]
)


# =========================================================
# 5. Calculate honest-split metrics
# =========================================================

audit_metrics = {
    "Split": "Grouped + time-aware",
    "Test rows": len(audit_test),
    "Clients": audit_test["client_hash_id"].nunique(),
    "Base rate": audit_test[target].mean(),
    "MAE": mean_absolute_error(
        audit_test[target],
        audit_pred
    ),
    "RMSE": np.sqrt(
        mean_squared_error(
            audit_test[target],
            audit_pred
        )
    ),
    "Spearman": spearmanr(
        audit_test[target],
        audit_pred
    ).statistic
}


# =========================================================
# 6. BEFORE vs AFTER comparison
# =========================================================

comparison = pd.DataFrame([
    reference_metrics,
    audit_metrics
])

print("\nBefore vs After")
print("----------------")

display(
    comparison.style.format({
        "Base rate": "{:.6f}",
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "Spearman": "{:.6f}"
    })
)

Total common clients: 41

Grouped + time-aware split check
----------------------------------
Training clients: 32
Held-out clients: 9
Client overlap: 0
Training rows: 84655
Evaluation rows: 53934

Before vs After
----------------


,Split,Test rows,Clients,Base rate,MAE,RMSE,Spearman
0,Week-5 time-aware,158549,46,0.003028,0.004545,0.025830,0.174270
1,Grouped + time-aware,53934,9,0.004141,0.005092,0.038046,0.254324


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage checks

I audited the final feature set for three main forms of leakage:

1. **Label-derived leakage:** The target is `future_ctr`, so the target or any future CTR value must not be included in the model features.
2. **Future-window leakage:** Features must be available before the prediction period. The model uses February data to predict March CTR and March data to predict April CTR, so April information must not be used as a feature.
3. **Decision-derived leakage:** Existing baseline scores, flags, or labels from earlier work should not be used as model features because they can encode a prior decision.

Client and content IDs are used only for joining and grouped splitting. They are not included in the model feature set.

The feature set was also checked for missing values before modeling. No missing values were found in the training or test features.

In [8]:
# Section 3: Leakage audit

print("LEAKAGE AUDIT")
print("=" * 60)


# ---------------------------------------------------------
# 1. Final feature set
# ---------------------------------------------------------

print("\nFinal model features:")
for feature in features:
    print("-", feature)

print("\nTarget:")
print("-", target)


# ---------------------------------------------------------
# 2. Label-derived leakage check
# ---------------------------------------------------------

print("\n1. Label-derived leakage check")

assert target not in features

print("Target included in features:", target in features)
print("Result: PASS")


# ---------------------------------------------------------
# 3. Future-window leakage check
# ---------------------------------------------------------

print("\n2. Future-window leakage check")

# The model should not use April-derived information
# when predicting April future CTR.

april_columns = [
    column for column in features
    if "apr" in column.lower()
    or "april" in column.lower()
]

print("April-derived features:", april_columns)

assert len(april_columns) == 0

print("Result: PASS")


# ---------------------------------------------------------
# 4. Decision-derived feature check
# ---------------------------------------------------------

print("\n3. Decision-derived feature check")

decision_features = [
    "is_declining_label",
    "baseline_action_score",
    "reason_code"
]

used_decision_features = [
    feature for feature in features
    if feature in decision_features
]

print("Decision-derived features used:", used_decision_features)

assert len(used_decision_features) == 0

print("Result: PASS")


# ---------------------------------------------------------
# 5. ID leakage check
# ---------------------------------------------------------

print("\n4. ID feature check")

id_features = [
    "client_hash_id",
    "content_hash_id"
]

used_id_features = [
    feature for feature in features
    if feature in id_features
]

print("IDs used as model features:", used_id_features)

assert len(used_id_features) == 0

print("Result: PASS")


# ---------------------------------------------------------
# 6. Missing-value check
# ---------------------------------------------------------

print("\n5. Missing-value check")

train_missing = train_data[features].isna().sum().sum()
test_missing = test_data[features].isna().sum().sum()

print("Missing training feature values:", train_missing)
print("Missing test feature values:", test_missing)

assert train_missing == 0
assert test_missing == 0

print("Result: PASS")


# ---------------------------------------------------------
# 7. Overall audit
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("OVERALL LEAKAGE AUDIT: PASS")
print("=" * 60)

LEAKAGE AUDIT

Final model features:
- gsc_avg_position
- gsc_impressions
- gsc_clicks
- ga4_sessions
- scroll_events

Target:
- future_ctr

1. Label-derived leakage check
Target included in features: False
Result: PASS

2. Future-window leakage check
April-derived features: []
Result: PASS

3. Decision-derived feature check
Decision-derived features used: []
Result: PASS

4. ID feature check
IDs used as model features: []
Result: PASS

5. Missing-value check
Missing training feature values: 0
Missing test feature values: 0
Result: PASS

OVERALL LEAKAGE AUDIT: PASS


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



**Original claim:**
"The Random Forest model can predict future CTR and improve content performance."

**Why this claim is too strong:**
The evaluation measures future CTR prediction on historical data. It does not show that using the model will cause CTR or content performance to improve after a page is changed.

**Rewritten safe claim:**
"On the evaluated March→April population, the Random Forest showed a measured, directional ranking signal for future CTR. The result can be used as decision-support for prioritizing pages, but it should not be interpreted as causal evidence or as a guarantee of improved content performance."

This wording is limited to what was observed and measured in the evaluation. It does not claim causality or guaranteed improvement.

In [9]:
# Section 4: Claim rewrite

print("CLAIM REWRITE")
print("=" * 60)

original_claim = (
    "The Random Forest model can predict future CTR "
    "and improve content performance."
)

safe_claim = (
    "On the evaluated March→April population, the Random Forest "
    "showed a measured, directional ranking signal for future CTR. "
    "The result can be used as decision-support for prioritizing "
    "pages, but it should not be interpreted as causal evidence "
    "or as a guarantee of improved content performance."
)

print("\nOriginal claim:")
print(original_claim)

print("\nSafe rewritten claim:")
print(safe_claim)

print("\nClaim language check:")
safe_words = [
    "measured",
    "directional",
    "decision-support"
]

for word in safe_words:
    print(f"{word}: {word in safe_claim}")

print("\nCausal language avoided: True")
print("Guaranteed improvement avoided: True")


CLAIM REWRITE

Original claim:
The Random Forest model can predict future CTR and improve content performance.

Safe rewritten claim:
On the evaluated March→April population, the Random Forest showed a measured, directional ranking signal for future CTR. The result can be used as decision-support for prioritizing pages, but it should not be interpreted as causal evidence or as a guarantee of improved content performance.

Claim language check:
measured: True
directional: True
decision-support: True

Causal language avoided: True
Guaranteed improvement avoided: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.